# Генерация кандидатов для поиска услуг Авито — улучшенное решение

Цель — для каждого поискового запроса вернуть 50 объявлений и максимизировать
`Recall@50`. Порядок внутри пятидесятки не важен, поэтому сначала собираем
широкий набор кандидатов несколькими способами, а затем выбираем из него 50.

Ноутбук показывает ход решения от простой baseline до гибрида с обучаемым
reranker. Все внешние модели и API исключены: решение воспроизводится локально
на CPU. Тяжёлые матрицы строятся последовательно, чтобы уложиться в 32 ГБ RAM.


## 0. Импорты и настройки

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
RANDOM_STATE = 42
TOP_K = 50
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## 1. Загрузка данных

Обычно файлы уже лежат рядом с ноутбуком в папке `data`. Следующая ячейка проверяет это. Если папка пуста, архив можно скачать по публичной ссылке из условия: для этого достаточно переключить `DOWNLOAD_IF_MISSING = True`. Эта часть нужна только один раз и не участвует в построении предсказаний.

In [ ]:
DOWNLOAD_IF_MISSING = False
PUBLIC_URL = 'https://disk.yandex.ru/d/sNhfo0YOjGtufg'
required_files = [
    DATA_DIR / 'train.parquet',
    DATA_DIR / 'benchmark_queries.parquet',
    DATA_DIR / 'benchmark_items.parquet',
]

if not all(path.exists() for path in required_files) and DOWNLOAD_IF_MISSING:
    import hashlib
    import requests
    import zipfile

    meta = requests.get(
        'https://cloud-api.yandex.net/v1/disk/public/resources',
        params={'public_key': PUBLIC_URL}, timeout=60
    ).json()
    resource = meta['_embedded']['items'][0]
    archive_path = DATA_DIR / 'dataset.zip'
    with requests.get(resource['file'], stream=True, timeout=120) as response:
        response.raise_for_status()
        with archive_path.open('wb') as output:
            for chunk in response.iter_content(2 ** 20):
                output.write(chunk)
    digest = hashlib.md5(archive_path.read_bytes()).hexdigest()
    assert digest == resource['md5'], 'Архив скачался с ошибкой'
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIR)

missing = [str(path) for path in required_files if not path.exists()]
assert not missing, f'Не найдены входные файлы: {missing}'
print('Все входные файлы найдены.')

In [ ]:
train = pd.read_parquet(DATA_DIR / 'train.parquet')
queries = pd.read_parquet(DATA_DIR / 'benchmark_queries.parquet')
items = pd.read_parquet(DATA_DIR / 'benchmark_items.parquet')

print(f'train:   {train.shape}')
print(f'queries: {queries.shape}')
print(f'items:   {items.shape}')

## 2. Первичная проверка и EDA

Смотрим типы данных, пропуски и несколько важных пересечений. Идентификаторы сразу проверяем как строки: это защищает от потери ведущих нулей.

In [ ]:
display(queries.head(3))
display(items[['item_id', 'item_title_raw', 'item_category_id', 'item_location_id']].head(3))

assert queries['query_id'].dtype.name in {'string', 'str', 'object'}
assert items['item_id'].dtype.name in {'string', 'str', 'object'}
assert queries['query_id'].is_unique
assert items['item_id'].is_unique

eda = pd.DataFrame({
    'rows': [len(train), len(queries), len(items)],
    'columns': [train.shape[1], queries.shape[1], items.shape[1]],
    'duplicates': [train.duplicated().sum(), queries.duplicated().sum(), items.duplicated().sum()],
}, index=['train', 'queries', 'items'])
display(eda)

In [ ]:
train_item_ids = set(train['item_id'])
train_query_texts = set(train['search_query'].fillna(''))

print('Уникальных текстов запросов в train:', train['search_query'].nunique())
print('Объявлений benchmark, встречавшихся в train:', items['item_id'].isin(train_item_ids).sum())
print('Текстов benchmark-запросов, встречавшихся в train:', queries['search_query'].isin(train_query_texts).sum())

missing_share = items.isna().mean().sort_values(ascending=False)
display(missing_share[missing_share > 0].to_frame('missing_share'))

### Наблюдения

* Категория запроса очень крупная и почти всегда одинакова, поэтому жёстко фильтровать только по ней мало полезно.
* Локация информативна, но услуги бывают выездными и удалёнными. Используем её как мягкий бонус, а не обязательный фильтр.
* Часть текстов запросов уже встречалась в обучении. Исторические клики дадут дополнительный канал, но только если соответствующее объявление есть в benchmark-корпусе.
* Для русского текста полезны одновременно словные признаки и символьные n-граммы: последние устойчивее к опечаткам, окончаниям и написанию слов слитно/раздельно.

## 3. Подготовка текста

In [ ]:
def normalize_text(value):
    """Небольшая нормализация без внешних морфологических библиотек."""
    if pd.isna(value):
        return ''
    value = unicodedata.normalize('NFKC', str(value)).lower().replace('ё', 'е')
    value = re.sub(r'[^0-9a-zа-я]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

def clean_series(series):
    return series.fillna('').map(normalize_text)

item_title = clean_series(items['item_title_raw'])
item_params = clean_series(items['item_infm_params_text'])
item_description = clean_series(items['item_description_raw']).str.slice(0, 800)

# Заголовок повторяем: он обычно точнее длинного описания.
item_word_text = item_title + ' ' + item_title + ' ' + item_params + ' ' + item_description
item_char_text = item_title + ' ' + item_params.str.slice(0, 250)

query_main = clean_series(queries['search_query'])
query_params = clean_series(queries['search_infm_params_text'])
query_word_text = query_main + ' ' + query_main + ' ' + query_params
query_char_text = query_main

pd.DataFrame({
    'query': query_word_text.head(3),
    'example_item': item_word_text.head(3),
})

## 4. Локальная валидация

У benchmark нет разметки. Для честной приблизительной проверки берём строки `train`, чьи выбранные объявления существуют в benchmark-корпусе. Одинаковые признаки запроса группируем: все связанные с ними объявления считаем релевантными.

Исторический lookup на этой выборке не оцениваем — он увидел бы правильные ответы и дал бы утечку. На валидации сравниваются только текстовые методы и мягкий бонус локации.

In [ ]:
query_cols = [
    'search_query', 'search_location_id', 'search_is_delivery_search',
    'search_infm_params_text', 'search_category'
]
benchmark_item_set = set(items['item_id'])
validation_rows = train.loc[train['item_id'].isin(benchmark_item_set), query_cols + ['item_id']].copy()
validation_rows['_group'] = pd.util.hash_pandas_object(
    validation_rows[query_cols].fillna(''), index=False
).astype(str)

relevant_by_group = validation_rows.groupby('_group')['item_id'].agg(lambda x: set(x))
validation_queries = validation_rows.drop_duplicates('_group').set_index('_group')
validation_queries['relevant'] = relevant_by_group

# Ограничиваем размер, чтобы ноутбук быстро выполнялся на обычном ноутбуке.
VALIDATION_SIZE = min(1500, len(validation_queries))
validation_queries = validation_queries.sample(VALIDATION_SIZE, random_state=RANDOM_STATE)
print('Псевдозапросов для валидации:', len(validation_queries))
print('Среднее число релевантных объявлений:', validation_queries['relevant'].map(len).mean().round(3))

## 5. Способ 1: словный TF-IDF

Словные униграммы и биграммы хорошо ловят точные названия услуг и устойчивые фразы. Матрица разреженная, поэтому весь корпус помещается в память.

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), min_df=2, max_df=0.995,
    max_features=180_000, sublinear_tf=True, dtype=np.float32, norm='l2'
)
item_word_matrix = word_vectorizer.fit_transform(item_word_text)
query_word_matrix = word_vectorizer.transform(query_word_text)
print('Word matrix:', item_word_matrix.shape, 'nnz:', item_word_matrix.nnz)

## 6. Способ 2: символьный TF-IDF

Символьные n-граммы дополняют словный поиск: запрос `видеодомофон` может совпасть с `видео домофон`, а небольшая опечатка не уничтожает всё совпадение.

In [ ]:
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5), min_df=3, max_df=0.995,
    max_features=160_000, sublinear_tf=True, dtype=np.float32, norm='l2'
)
item_char_matrix = char_vectorizer.fit_transform(item_char_text)
query_char_matrix = char_vectorizer.transform(query_char_text)
print('Char matrix:', item_char_matrix.shape, 'nnz:', item_char_matrix.nnz)

## 7. Быстрый поиск top-k по разреженным матрицам

In [ ]:
def sparse_topk(query_matrix, item_matrix, top_k=100, batch_size=64):
    """Возвращает индексы top-k документов без создания огромной плотной матрицы."""
    result = []
    item_matrix_t = item_matrix.T.tocsc()
    for start in tqdm(range(0, query_matrix.shape[0], batch_size), desc='retrieval'):
        scores = (query_matrix[start:start + batch_size] @ item_matrix_t).tocsr()
        for row_idx in range(scores.shape[0]):
            left, right = scores.indptr[row_idx], scores.indptr[row_idx + 1]
            values = scores.data[left:right]
            indices = scores.indices[left:right]
            if len(values) > top_k:
                keep = np.argpartition(values, -top_k)[-top_k:]
                keep = keep[np.argsort(values[keep])[::-1]]
                indices = indices[keep]
            elif len(values):
                indices = indices[np.argsort(values)[::-1]]
            result.append(indices.tolist())
    return result

def rrf(lists, weights=None, k=60, limit=100):
    """Reciprocal Rank Fusion: объединяет ранги методов с разными шкалами score."""
    if weights is None:
        weights = [1.0] * len(lists)
    scores = defaultdict(float)
    for ranking, weight in zip(lists, weights):
        for rank, doc_idx in enumerate(ranking):
            scores[doc_idx] += weight / (k + rank + 1)
    return [idx for idx, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:limit]]

def add_location_bonus(ranking, query_location, bonus=0.004, limit=50):
    scores = {idx: 1.0 / (60 + rank + 1) for rank, idx in enumerate(ranking)}
    for idx in ranking:
        if items.iloc[idx]['item_location_id'] == query_location:
            scores[idx] += bonus
    return [idx for idx, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:limit]]

def recall_at_k(predicted_indices, relevant_ids, k=50):
    predicted_ids = set(items.iloc[predicted_indices[:k]]['item_id'])
    return len(predicted_ids & relevant_ids) / len(relevant_ids)

## 8. Сравнение методов на локальной валидации

In [ ]:
val_main = clean_series(validation_queries['search_query'])
val_params = clean_series(validation_queries['search_infm_params_text'])
val_word_text = val_main + ' ' + val_main + ' ' + val_params

val_word_matrix = word_vectorizer.transform(val_word_text)
val_char_matrix = char_vectorizer.transform(val_main)
val_word_rankings = sparse_topk(val_word_matrix, item_word_matrix, top_k=120)
val_char_rankings = sparse_topk(val_char_matrix, item_char_matrix, top_k=120)

validation_scores = defaultdict(list)
for pos, (_, row) in enumerate(validation_queries.iterrows()):
    word_rank = val_word_rankings[pos]
    char_rank = val_char_rankings[pos]
    hybrid_rank = rrf([word_rank, char_rank], weights=[1.0, 0.8], limit=120)
    hybrid_location_rank = add_location_bonus(hybrid_rank, row['search_location_id'], limit=TOP_K)

    validation_scores['word TF-IDF'].append(recall_at_k(word_rank, row['relevant']))
    validation_scores['char TF-IDF'].append(recall_at_k(char_rank, row['relevant']))
    validation_scores['hybrid RRF'].append(recall_at_k(hybrid_rank, row['relevant']))
    validation_scores['hybrid + location'].append(recall_at_k(hybrid_location_rank, row['relevant']))

score_table = pd.Series({name: np.mean(values) for name, values in validation_scores.items()})
display(score_table.sort_values(ascending=False).to_frame('Recall@50'))

## 9. Способ 3: история кликов

Для повторяющихся нормализованных запросов берём объявления, которые уже выбирали пользователи. В ответ могут попасть только идентификаторы из `benchmark_items`. Частоту используем как порядок, но исторический список не заменяет текстовый поиск, а лишь добавляется в общий RRF.

In [ ]:
item_index_by_id = pd.Series(np.arange(len(items)), index=items['item_id']).to_dict()
history = train.loc[train['item_id'].isin(benchmark_item_set), ['search_query', 'item_id']].copy()
history['normalized_query'] = clean_series(history['search_query'])
history_counts = history.groupby(['normalized_query', 'item_id']).size().rename('clicks').reset_index()
history_counts = history_counts.sort_values(['normalized_query', 'clicks'], ascending=[True, False])
history_lookup = history_counts.groupby('normalized_query')['item_id'].agg(list).to_dict()

history_coverage = query_main.isin(history_lookup).mean()
print(f'Исторические кандидаты есть для {history_coverage:.1%} benchmark-запросов.')

## 10. Финальное объединение

Объединяем три списка с помощью RRF. Вес истории выше, потому что это наблюдавшееся действие пользователя, но текстовые методы сохраняют большую часть мест и помогают новым запросам. Затем применяем небольшой бонус объявлениям из той же локации. Если совпадений неожиданно мало, дополняем ответ популярными доступными объявлениями.

In [ ]:
word_rankings = sparse_topk(query_word_matrix, item_word_matrix, top_k=140)
char_rankings = sparse_topk(query_char_matrix, item_char_matrix, top_k=140)

# Универсальный fallback: объявления с большим числом отзывов и высоким рейтингом.
popularity = (
    items['item_rating_reviews_count'].fillna(0).map(np.log1p)
    * items['item_rating'].fillna(0)
)
fallback_indices = popularity.sort_values(ascending=False).index.tolist()

final_rankings = []
for pos, row in tqdm(queries.iterrows(), total=len(queries), desc='fusion'):
    historical_ids = history_lookup.get(query_main.iloc[pos], [])[:100]
    historical_indices = [item_index_by_id[item_id] for item_id in historical_ids]

    combined = rrf(
        [word_rankings[pos], char_rankings[pos], historical_indices],
        weights=[1.0, 0.8, 2.0], limit=160
    )
    combined = add_location_bonus(combined, row['search_location_id'], limit=TOP_K)

    # Обычно список уже содержит 50 элементов; fallback нужен лишь для пустых/аномальных текстов.
    seen = set(combined)
    if len(combined) < TOP_K:
        for idx in fallback_indices:
            if idx not in seen:
                combined.append(idx)
                seen.add(idx)
            if len(combined) == TOP_K:
                break
    final_rankings.append(combined[:TOP_K])

### Посмотрим несколько результатов вручную

In [ ]:
for query_pos in range(3):
    print('\nQUERY:', queries.iloc[query_pos]['search_query'])
    display(items.iloc[final_rankings[query_pos][:5]][['item_id', 'item_title_raw', 'item_location_id']])

## 11. Сохранение baseline и строгая проверка формата

In [ ]:
predictions = [items.iloc[ranking]['item_id'].astype(str).tolist() for ranking in final_rankings]
answer = pd.DataFrame({
    'query_id': queries['query_id'].astype(str),
    'answer': [' '.join(item_ids) for item_ids in predictions],
})
answer.to_csv('answer_baseline.csv', index=False, encoding='utf-8')
display(answer.head())

In [ ]:
# Проверки формата из условия.
valid_item_ids = set(items['item_id'].astype(str))
saved = pd.read_csv('answer_baseline.csv', dtype=str, keep_default_na=False)

assert saved.columns.tolist() == ['query_id', 'answer']
assert len(saved) == len(queries)
assert saved['query_id'].tolist() == queries['query_id'].astype(str).tolist()
assert saved['query_id'].is_unique

for query_id, value in saved.itertuples(index=False):
    ids = value.split(' ')
    assert 1 <= len(ids) <= TOP_K, query_id
    assert len(ids) == len(set(ids)), f'Повтор в строке {query_id}'
    assert set(ids) <= valid_item_ids, f'Неизвестный item_id в строке {query_id}'
    assert all(re.fullmatch(r'[0-9a-f]{16}', item_id) for item_id in ids), query_id

print(f'Baseline готов: {len(saved)} строк, все проверки пройдены.')

## Промежуточный итог

Baseline объединяет word TF-IDF, char TF-IDF, историю кликов и мягкий exact
location bonus. Первая отправка этого семейства получила `Recall@50 = 0.382072`.
Дальше проверим, какие сигналы действительно дают прирост и можно ли выбирать
лучшие 50 из более широкого union кандидатов.

## 12. Глубокий EDA: что специфично в этих данных

Отдельный memory-safe анализ находится в `eda_deep.py`, полный текст — в
`eda_report.md`. Самые полезные для модели наблюдения:

- в train 73 706 нормализованных текстов запроса, но 354 241 полных контекстов;
- только 9.59% объявлений benchmark встречались среди исторических кликов;
- 37.48% benchmark-запросов встречались по тексту, но лишь 4.45% — полным
  контекстом: lookup не может быть основным решением;
- exact location совпадает у 83.10% обучающих пар, однако переходы между
  локациями покрывают ещё значимую долю;
- хотя полный OOV низкий (1.29% по объединению полей), точный token overlap с
  заголовком слабее, чем с описанием;
- почти половина строк корпуса имеет повторяющийся нормализованный заголовок,
  поэтому одного title недостаточно.

Вывод: нужны одновременно широкий текстовый retrieval, географический prior,
понимание микрокатегории и история — но как мягкие сигналы, а не фильтры.

In [ ]:
# Краткая таблица с результатами уже посчитанного EDA.
from pathlib import Path

eda_text = Path('eda_report.md').read_text(encoding='utf-8')
print(eda_text[:5000])  # полный отчёт удобнее открыть как markdown-файл

## 13. Честная локальная проверка

Для сравнения гибридов берём клики по объектам, которые присутствуют в
benchmark-корпусе, и группируем их по полному контексту запроса. Затем делим
контексты на `tune/test`.

Критически важная деталь: таблицы переходов локаций и профили микрокатегорий
строятся только по train-строкам, чьи объявления **не входят** в benchmark.
Иначе релевантный `item_id` мог бы незаметно утечь в lookup.

Reranker не получает сам `item_id`; он видит только 17 переносимых признаков:
rank/cosine пяти текстовых каналов, географию, микрокатегорию, рейтинг и число
отзывов.

In [ ]:
import json
import pandas as pd

experiment = json.loads(Path('hybrid_results.json').read_text(encoding='utf-8'))
results = pd.DataFrame(experiment['results'])
main_rows = [
    'mixed', 'char', 'mixed+char',
    'mixed+char + location+microcats+pop',
    'best + exact-location quota 40',
    'logistic supervised reranker',
    'logistic reranker + exact-location quota 20',
    'logistic + leakage-safe history RRF weight 1.0',
    'union@all-channels oracle',
]
display(
    results[(results.phase == 'test') & results.name.isin(main_rows)]
    .drop_duplicates('name', keep='last')
    .sort_values('macro_recall50')
    .reset_index(drop=True)
)

## 14. Несколько способов поиска и их объединение

В широкий union входят пять retrieval-каналов по 180 кандидатов:

1. `mixed`: запрос против `2 × title + params + description`;
2. отдельный `title`;
3. отдельные параметры;
4. отдельное описание;
5. char 3–5-граммы по title и params.

Простой RRF хорош как baseline, потому что не требует сравнивать cosine разных
пространств. Но он теряет информацию: первое место в слабом поле и первое место
в сильном поле получает почти одинаковый голос. Поэтому поверх union обучаем
маленький `SGDClassifier(log_loss)`. Он фактически служит быстрым L1-reranker и
оценивает каждую пару «запрос — объявление».

Полный union на holdout содержит 61.4% релевантных объектов, а обученная модель
возвращает 58.0% в top-50. Значит, она выбирает почти весь доступный signal и
основной следующий резерв — улучшать сам retrieval.

In [ ]:
coef = pd.DataFrame({
    'feature': experiment['reranker']['feature_order'],
    'coefficient': experiment['reranker']['coef'],
})
coef['abs_coefficient'] = coef.coefficient.abs()
display(coef.sort_values('abs_coefficient', ascending=False).drop(columns='abs_coefficient'))

### Как читать коэффициенты

Сильнейший сигнал — вероятность перехода от локации запроса к локации
объявления. Следом идут char rank и cosine смешанного текста. Это согласуется с
EDA и лучше бинарного правила «локации равны».

Знаки отдельных коррелирующих признаков нельзя трактовать причинно. Например,
raw rating получает отрицательный вес рядом с положительными popularity и
числом отзывов: модель различает «5.0 по одному отзыву» и устойчивый рейтинг.


## 15. Финальная схема

```text
word/char retrieval (5 × top-180)
              ↓ union
linear reranker по 17 признакам
              ↓ top-180
RRF с историческими кликами (вес 1.0)
              ↓
небольшая exact-location quota (до 20)
              ↓
50 item_id
```

История добавляется мягко. На leakage-safe holdout это лучше жёсткой вставки, а
для новых формулировок всё решение продолжает работать без истории.

## 16. Генерация `answer_v3.csv`

Полная реализация вынесена в `generate_v3.py`, чтобы не дублировать 300 строк и
чтобы генератор можно было запускать из терминала. Это не скрытая модель:
скрипт содержит те же функции TF-IDF, sparse top-k, признаки, RRF и проверки,
которые разобраны выше.

Перед запуском освобождаем baseline-матрицы. Генерация занимает несколько минут
на CPU; одновременно запускать второй тяжёлый процесс не следует.

In [ ]:
import gc
import subprocess
import sys

# Освобождаем крупные объекты baseline перед отдельным memory-safe прогоном.
for name in [
    'train', 'queries', 'items', 'item_word_matrix', 'query_word_matrix',
    'item_char_matrix', 'query_char_matrix', 'word_vectorizer', 'char_vectorizer',
    'word_rankings', 'char_rankings', 'final_rankings', 'item_word_text',
    'item_char_text', 'query_word_text', 'query_main', 'query_params',
]:
    globals().pop(name, None)
gc.collect()

subprocess.run([sys.executable, '-u', 'generate_v3.py'], check=True)

In [ ]:
# Независимая финальная проверка файла.
import re
import pandas as pd

queries_check = pd.read_parquet('data/benchmark_queries.parquet', columns=['query_id'])
items_check = pd.read_parquet('data/benchmark_items.parquet', columns=['item_id'])
valid_ids = set(items_check.item_id.astype(str))
submission = pd.read_csv('answer_v3.csv', dtype=str, keep_default_na=False)

assert submission.columns.tolist() == ['query_id', 'answer']
assert len(submission) == len(queries_check) == 2452
assert submission.query_id.tolist() == queries_check.query_id.astype(str).tolist()
assert submission.query_id.is_unique
for query_id, answer_string in submission.itertuples(index=False):
    ids = answer_string.split(' ')
    assert len(ids) == 50
    assert len(ids) == len(set(ids))
    assert set(ids) <= valid_ids
    assert all(re.fullmatch(r'[0-9a-f]{16}', item_id) for item_id in ids), query_id

print('answer_v3.csv готов и прошёл все проверки')
display(submission.head(3))

## 17. Итог и ограничения

Лучший локальный результат — `Recall@50 ≈ 0.580` против `0.382072` у первой
leaderboard-отправки. Это не обещание такого же публичного score: локальный
holdout сформирован из train, временных меток нет, а настоящих показов/негативов
в данных тоже нет.

Тем не менее улучшение опирается не на подбор конкретных `query_id`, а на
переносимые признаки и отдельный test split. Следующей проверяемой гипотезой был
бы локальный sentence-embedding retrieval или BM25, но текущий вариант уже
сохраняет хороший баланс качества, прозрачности и требований к памяти.